# Answering Questions About Iceberg Data Using Pydough CE with a BodoSQL Backend

This notebook demonstrates how to use Pydough and an LLM to answer questions about data stored in Iceberg table format, with BodoSQL as the engine.

BodoSQL is a high performance, scalable SQL engine that can query data from a variety of sources. This example uses Iceberg tables written as local files. 

Apache Iceberg is an open source table format that adds a layer of scalable dataset management, including ACID transactions, evolving schemas, and time travel, on top of raw files.  

The example used in this notebook was derived from https://github.com/bodo-ai/PyDough/blob/main/demos/notebooks/BODOSQL_demo.ipynb with modifications to demonstrate LLM integration and Iceberg data. 

This notebook has the following sections:

1. Installation & Setup
    
    Import packages, set environment variables and generate data.

2. Question Answering using an LLM & Pydough with BodoSQL as the Backend

    Ask natural language questions about the generated dataset and get results using BodoSQL as the execution engine.

3. A Closer Look at the BodoSQL backend

    This section takes a closer look at how the BodoSQL backend functions under the hood, highlighting important optimizations that makes it well suited for querying massive datasets.


## 1. Installation & Setup

The following line installs the required dependencies for this notebook: Pydough-analytics, Pydough, BodoSQL and Pyiceberg:
``` shell
pip install ".[bodosql,iceberg,notebooks]"
```

For this notebook, you will also need to install any additional packages required to use your LLM. 


In [1]:
import json
import datetime
import os
os.environ["BODO_NUM_WORKERS"] = "1"
import shutil

from dotenv import load_dotenv
import bodo.pandas as bd
import numpy as np
import numpy.typing as npt
import pandas as pd
import pydough
from bodosql import BodoSQLContext, FileSystemCatalog
from pydough_analytics.llm.llm_client import LLMClient
from pydough_analytics.commands.generate_md_cmd import generate_markdown_from_config

### Environment Setup

Configure your environment based on the LLM and provider you are using:

To use the default option, Google's Agent Platform (formerly Vertex AI) with Gemini 2.5 Pro, you will need to set the following environment variables:
``` bash
GOOGLE_PROJECT_ID="YOUR_PROJECT_ID"
GOOGLE_GENAI_USE_VERTEXAI=true
```

And login / configure your project with:
``` bash
gcloud auth login
gcloud config set project YOUR_PROJECT_ID
```

You can also use a different provider and LLM. For example, to use this notebook with openai, run `pip install openai` and set your `OPENAI_API_KEY`. 


In [2]:
load_dotenv(dotenv_path="../../.env", override=True)

True

In [3]:
# TODO: Configure your own model and provider
provider="openai"
model="gpt-5.5"

### Generating the Data

For this example, we will be generating a synthetic, "colorshop" database using the following code:

In [4]:
datapath = "../data/datasets/colors.csv"

def generate_color_tables():
    color_df: pd.DataFrame = pd.read_csv(
        datapath,
        names=["IDENTNAME", "COLORNAME", "CHEX", "R", "G", "B"],
    )
    customers_df: pd.DataFrame = pd.DataFrame(
        {
            "CSTID": range(1, 21),
            "CSTFNAME": [
                "Alice",
                "Bob",
                "Charlie",
                "David",
                "Eve",
                "Frank",
                "Grace",
                "Heidi",
                "Ivan",
                "Janet",
                "Karl",
                "Leo",
                "Marcel",
                "Nina",
                "Oscar",
                "Peggy",
                "Quentin",
                "Rahul",
                "Sybil",
                "Trent",
            ],
            "CSTLNAME": [
                "Smith",
                "Johnson",
                "Williams",
                "Jones",
                "Brown",
                "Davis",
                "Miller",
                "Wilson",
                "Moore",
                "Taylor",
                "Anderson",
                "Thomas",
                "Jackson",
                "White",
                "Harris",
                "Martin",
                "Sharma",
                "Garcia",
                "Martinez",
                "Robinson",
            ],
        }
    )
    suppliers_df: pd.DataFrame = pd.DataFrame(
        {
            "SUPID": range(1, 6),
            "SUPNAME": [
                "Pallette Emporium",
                "Rainbow Inc.",
                "Hue Depot",
                "Chroma Co.",
                "Tint Traders",
            ],
        }
    )
    # Generate the shipments table with 200,000 rows using random generation
    # while maintaining referential integrity with the other three tables and
    # creating some correlations/trends in the data to make the queries more
    # interesting. The random seed is fixed to ensure test reproducibility.
    rng: np.random.Generator = np.random.default_rng(seed=42)
    n_shipments: int = 200000
    color_indices: npt.NDArray[np.int_] = np.minimum(
        rng.integers(len(color_df), size=n_shipments),
        rng.integers(len(color_df), size=n_shipments),
    )
    customer_ids: npt.NDArray[np.int_] = np.minimum(
        rng.integers(1, len(customers_df) + 1, size=n_shipments),
        rng.integers(1, 2 * len(customers_df), size=n_shipments),
    )
    supplier_ids: npt.NDArray[np.int_] = np.minimum(
        rng.integers(1, len(suppliers_df) + 1, size=n_shipments),
        rng.integers(1, round(1.5 * len(suppliers_df)), size=n_shipments),
    )
    dates_of_shipment: list[datetime.date] = sorted(
        [
            datetime.date.fromordinal(738886 + i)
            for i in rng.integers(770, size=n_shipments)
        ]
    )
    volumes: npt.NDArray[np.float64] = rng.choice(
        [0.5, 1.0, 2.0, 4.5, 10.0], p=[0.1, 0.4, 0.3, 0.15, 0.05], size=n_shipments
    )
    prices: npt.NDArray[np.float64] = np.round(
        volumes * (12 + (((1 + color_indices) * (1 + supplier_ids)) % 11.11)), 2
    )
    shipments_df: pd.DataFrame = pd.DataFrame(
        {
            "SID": range(n_shipments),
            "COLID": color_df.loc[color_indices, "IDENTNAME"].values,
            "CUSID": customer_ids,
            "COMID": supplier_ids,
            "DOS": dates_of_shipment,
            "VOL": volumes,
            "PRC": prices,
        }
    )
    return color_df, customers_df, suppliers_df, shipments_df

color_df, customers_df, suppliers_df, shipments_df = generate_color_tables()

### Inspecting the data

The example dataset consists of 4 tables: `COLOR`,`CUST`, `SUPL` and `SHIP`: 
* The `COLOR` table consists of color identifier and color name, as well as the hex value and R,G,B values of the color. 
* The `CUST` table consists of customer name and ID.
* The `SUPL` table consists of supplier name and ID.
* The `SHIP` table consists of a shipment ID along with the color identifier, customer ID, supplier/company ID as well as the date, amount of paint purchased, and price. 

In [5]:
for df in [color_df, customers_df, suppliers_df, shipments_df]:
    display(df.head(), len(df))

,IDENTNAME,COLORNAME,CHEX,R,G,B
0,air_force_blue_raf,Air Force Blue (Raf),#5d8aa8,93,138,168
1,air_force_blue_usaf,Air Force Blue (Usaf),#00308f,0,48,143
2,air_superiority_blue,Air Superiority Blue,#72a0c1,114,160,193
3,alabama_crimson,Alabama Crimson,#a32638,163,38,56
4,alice_blue,Alice Blue,#f0f8ff,240,248,255


865

,CSTID,CSTFNAME,CSTLNAME
0,1,Alice,Smith
1,2,Bob,Johnson
2,3,Charlie,Williams
3,4,David,Jones
4,5,Eve,Brown


20

,SUPID,SUPNAME
0,1,Pallette Emporium
1,2,Rainbow Inc.
2,3,Hue Depot
3,4,Chroma Co.
4,5,Tint Traders


5

,SID,COLID,CUSID,COMID,DOS,VOL,PRC
0,0,boston_university_red,16,2,2024-01-01,1.0,12.69
1,1,redwood,9,2,2024-01-01,2.0,44.40
2,2,pale_cerulean,6,3,2024-01-01,1.0,13.56
3,3,indigo_dye,8,1,2024-01-01,1.0,16.52
4,4,inchworm,9,1,2024-01-01,4.5,79.34


200000

### Write the data as Iceberg tables

Next, we will convert the generated dataset to local Iceberg tables using Bodo's Directory catalog.
Bodo also supports S3Tables, and REST catalogs such as Apache Polaris. 

To see a full list of supported catalogs, refer to Bodo/BodoSQL's [Iceberg documentation](https://docs.bodo.ai/latest/api_docs/sql/database_catalogs/#supported-query-types_4). 

In [6]:
from pyiceberg.partitioning import PartitionField, PartitionSpec
from pyiceberg.transforms import IdentityTransform, BucketTransform

iceberg_warehouse = "iceberg_warehouse"

table_names = ["COLOR", "CUST", "SUPL", "SHIP"]
tables = [color_df, customers_df, suppliers_df, shipments_df]

if not os.path.exists(iceberg_warehouse):
    bd.from_pandas(color_df).to_iceberg("COLOR", location=iceberg_warehouse)
    bd.from_pandas(customers_df).to_iceberg("CUST", location=iceberg_warehouse)
    bd.from_pandas(suppliers_df).to_iceberg("SUPL", location=iceberg_warehouse)

    # Partition shipments table by company, color
    spec = PartitionSpec(
        PartitionField(4, 1001, IdentityTransform(), "com_part"),
        PartitionField(2, 1002, BucketTransform(10), "col_part")
    )
    bd.from_pandas(shipments_df).to_iceberg("SHIP", location=iceberg_warehouse, partition_spec=spec)

### Create a BodoSQL context

Next, we create a BodoSQL context and connect to our Iceberg database. For information on creating a catalog from different database types, [see here](https://docs.bodo.ai/latest/api_docs/sql/database_catalogs/).

To create a BodoSQL context from Iceberg tables written to local files in the step above, use `FilesystemCatalog`:

In [7]:
catalog = FileSystemCatalog(os.path.abspath(iceberg_warehouse))
bc = BodoSQLContext(catalog=catalog)

### Create Pydough Metadata

There are two metadata files that we need in order to use Pydough with an LLM: the knowledge graph, written in JSON format, which is used by the Pydough DSL to convert Pydough code to SQL, and a markdown file derived from the knowledge graph for sharing metadata with the LLM. 

The knowledge graph contains details about your tables as well as the relationships between them. You can find the complete spec for creating a Pydough knowledge graph [here](https://github.com/bodo-ai/PyDough/blob/main/documentation/metadata.md). In this example, we demonstrate how you might generate a knowledge graph for your own data using a simple Python script. We also provide a pre-built metadata graph for this dataset. 

The markdown file contains the same information as the knowledge graph in a format that is more suitable for prompting an LLM. Pydough-analytics provides a function: `generate_markdown_from_config()` for creating a markdown file from a Pydough knowledge graph.  

In [8]:
PREGENERATED_KG_PATH = "../data/metadata/Colorshop_graph.json"
PREGENERATED_MD_PATH = "../data/metadata_markdowns/Colorshop.md"

db_name = "COLORSHOP"
kg_path = PREGENERATED_KG_PATH
md_path = PREGENERATED_MD_PATH

generate_markdown_from_config(db_name, json_path=kg_path, md_path=md_path)

Markdown written to ../data/metadata_markdowns/Colorshop.md


The following code demonstrates how you could construct an equivalent Pydough knowledge graph using Python dictionaries. 

In [ ]:
def create_simple_join_relationship(name, reverse_name, parent, child, keys):
    """ Adds a simple join relationship as well as reverse relationship. """
    forward_direction = {
        "type": "simple join",
        "name": name,
        "parent collection": parent,
        "child collection": child,
        "keys": keys,
        "singular": False,
        "always matches": False
    }

    reverse_rel = {
        "type": "reverse",
        "name": reverse_name,
        "original parent": parent,
        "original property": name,
        "singular": True,
        "always matches": True,
    }


    return [forward_direction, reverse_rel]


def create_simple_table(table, name, unique_properties, column_name_map, property_types_map):
    """ Adds a simple table with no relationships. """
    properties = []
    for col in table.columns:
        properties.append(
            {
                "name": column_name_map.get(col, col).lower(),
                "type": "table column",
                "column name": col,
                "data type": property_types_map.get(col, "string"),
                "sample values": [str(val) for val in table[col].head().to_list()]
            }
        )

    table = {
        "name": name.lower(),
        "unique properties": unique_properties,
        "type": "simple table",
        "table path": name,
        "properties": properties
    }

    return table


def create_graph(db_name, relationships, collections):
    """ Creates a graph config dictionary from the provided components. """
    return {
        "name": db_name,
        "version": "V2",
        "collections": collections,
        "relationships": relationships
    }


In [10]:
generated_kg_path = "generated_color_graph.json"
generated_md_path = "generated_color.md"

collections = []
relationships = []

tables_map = {
    "COLOR": color_df,
    "CUST": customers_df,
    "SUPL": suppliers_df,
    "SHIP": shipments_df
}

unique_properties = {
    "COLOR": ["identname", "colorname"],
    "CUST": ["cstid"],
    "SUPL": ["supid"],
    "SHIP": ["sid"]
}

properties_types = {
    "COLOR" : {
        "IDENTNAME": "string",
        "COLORNAME": "string",
        "CHEX": "string",
        "R": "numeric",
        "G": "numeric",
        "B": "numeric"
    },
    "CUST" : {
        "CSTID": "numeric",
        "CSTFNAME": "string",
        "CSTLNAME": "string"
    },
    "SUPL" : {
        "SUPID": "numeric",
        "SUPNAME": "string"
    },
    "SHIP" : {
        "SID": "numeric",
        "COLID": "string",
        "CUSID": "numeric",
        "COMID": "numeric",
        "DOS": "datetime",
        "VOL": "numeric",
        "PRC": "numeric"
    }
}

for table_name, table in tables_map.items():
    collection = create_simple_table(
        table,
        name=table_name,
        unique_properties=unique_properties[table_name],
        column_name_map=dict(),
        property_types_map=properties_types[table_name]
    )
    collections.append(collection)


relationships.extend(create_simple_join_relationship(
    name="shipments",
    reverse_name="colors",
    parent="color",
    child="ship",
    keys={"identname": ["colid"]},
))
relationships.extend(create_simple_join_relationship(
    name="shipments",
    reverse_name="customers",
    parent="cust",
    child="ship",
    keys={"cstid": ["cusid"]},
))
relationships.extend(create_simple_join_relationship(
    name="shipments",
    reverse_name="suppliers",
    parent="supl",
    child="ship",
    keys={"supid": ["comid"]},
))


graph = [create_graph(db_name, relationships, collections)]

json.dump(graph, open(generated_kg_path, "w"), indent=4)

In [11]:
generate_markdown_from_config(db_name, json_path=generated_kg_path, md_path=generated_md_path)

Markdown written to generated_color.md


In [ ]:
# Uncomment these lines to use the programmatically generated graph and markdown file.
# kg_path = generated_kg_path
# md_path = generated_md_path

## 2. Ask an LLM to generate Pydough code and execute with BodoSQL

We are now ready to start asking questions about our data in natural language! We will first use the `simple_ask` function, which returns a result object containing Pydough code and an explanation. This code needs to be run in an existing Pydough context. 

Then, we use the `ask` function, which creates the Pydough context, executes code, and returns a result object with a DataFrame as well as the generated SQL, in addition to the Pydough code and explanation.

In [12]:
client = LLMClient(
    provider=provider,
    model=model
)

question = "Find the 3 paint colors Quentin Sharma purchased the most of and how much volume was purchased for each."

In [10]:
with open(md_path, "r") as f:
    md_content = f.read()

result = client.simple_ask(
    question=question,
    md_content=md_content,
    db_name=db_name,
)

In [ ]:
if result.code is None:
    print(result.exception)
else:
    print(result.code)
    print(result.full_explanation)

In [ ]:
pydough.active_session.load_metadata_graph(kg_path, db_name)
pydough.active_session.connect_database("bodosql", context=bc)

# !Note: may need to adjust final variable name based on output
final_variable = "result"

pydough_str = pydough.from_string(result.code, answer_variable=final_variable)
display(pydough.to_df(pydough_str))

In [19]:
result = client.ask(
    question=question,
    kg_path=kg_path,
    md_path=md_path,
    db_name=db_name,
    bodosql_context=bc,
)

In [20]:
if result.df is None:
    print(result.exception)
else:
    display(result.df)

,PAINT_COLOR_NAME,TOTAL_PURCHASED_VOLUME
0,Flamingo Pink,66.5
1,Beau Blue,65.0
2,Dark Powder Blue,62.5


In [21]:
print(result.code)
print(result.full_explanation)

quentin_sharma_shipments = shipments.WHERE(
    (customer.first_name == "Quentin") & (customer.last_name == "Sharma")
).CALCULATE(
    paint_color_name=color.name
)

result = quentin_sharma_shipments.PARTITION(
    name="color_purchase_groups",
    by=paint_color_name
).CALCULATE(
    paint_color_name=paint_color_name,
    total_purchased_volume=SUM(shipments.volume)
).TOP_K(
    3,
    by=total_purchased_volume.DESC()
)
This code:

1. Starts from the `shipments` collection, since purchases are represented as shipments.

2. Filters to shipments purchased by the customer whose first name is `"Quentin"` and last name is `"Sharma"`.

3. Calculates the shipped color’s name as `paint_color_name`.

4. Groups Quentin Sharma’s shipments by paint color.

5. Sums the `volume` purchased for each color.

6. Returns the top 3 colors by total purchased volume.


In addition to the generated Pydough code, we can also look at the SQL query BodoSQL received from Pydough.

In [80]:
print(result.sql)

WITH _s3 AS (
  SELECT
    ship.colid,
    SUM(ship.vol) AS sum_vol
  FROM ship AS ship
  JOIN cust AS cust
    ON cust.cstfname = 'Quentin' AND cust.cstid = ship.cusid AND cust.cstlname = 'Sharma'
  GROUP BY
    1
)
SELECT
  color.colorname AS paint_color,
  COALESCE(_s3.sum_vol, 0) AS volume_purchased
FROM color AS color
JOIN _s3 AS _s3
  ON _s3.colid = color.identname
ORDER BY
  2 DESC NULLS LAST
LIMIT 3


### Verify results are correct

The following code snippet is a valid interpretation of the question that also fetches the volumes of paint purchased. We will use the pregenerated knowledge graph for consistent naming. 

In [22]:
pydough_str = """
selected_shipments = shipments.WHERE(
    (customer.first_name == "Quentin") &
    (customer.last_name == "Sharma")
)

output = (
    colors
    .CALCULATE(name, total_quantity=SUM(selected_shipments.volume))
    .TOP_K(3, by=total_quantity.DESC())
)
"""

pydough.active_session.connect_database("bodosql", context=bc)
pydough.active_session.load_metadata_graph(PREGENERATED_KG_PATH, db_name)

pydough.to_df(pydough.from_string(pydough_str, answer_variable="output"))

,NAME,TOTAL_QUANTITY
0,Flamingo Pink,66.5
1,Beau Blue,65.0
2,Dark Powder Blue,62.5


## 3. A Closer Look at the BodoSQL backend

In this example, we will look at some optimizations BodoSQL does under the hood.

BodoSQL includes a sophisticated query optimizer as well as a vectorized, parallel runtime based on MPI, allowing it to scale to massive datasets seamlessly. 
At the same time, Iceberg's aggregated metadata allows query engines like BodoSQL to skip over groups of files depending on query selectiveness, greatly improving IO performance especially when reading from object storage such as Amazon S3.

The following example answers the question:
 
"Which light-colored paint shipments generated the highest revenue for Pallette Emporium and Rainbow Inc.? For each shipment, provide the customer, price, color and company name."
 
Here, "light-colored paint shipments" refers to shipments where red, blue and green values are all >=100.

In [23]:
pydough.active_session.connect_database("bodosql", context=bc)
pydough.active_session.load_metadata_graph(PREGENERATED_KG_PATH, db_name)

PyDoughMetadata[graph 'COLORSHOP']

In [24]:
pydough_str = """
selected_shipments = shipments.WHERE(
    ISIN(company.name, ["Pallette Emporium", "Rainbow Inc."]) &
    (color.blue >= 100) & (color.green >= 100) & (color.red >= 100)
)

result = selected_shipments.CALCULATE(
    shipment_id=key,
    customer_name=JOIN_STRINGS(' ', customer.first_name, customer.last_name),
    color_name=color.name,
    company_name=company.name,
    price=price,
).TOP_K(
    5,
    by=price.DESC(),
)
"""

q = pydough.from_string(
    pydough_str, answer_variable="result"
)


Let's inspect the SQL generated from this code as well the logical plan generated by BodoSQL:

In [25]:
print(pydough.to_sql(q))

SELECT
  ship.sid AS shipment_id,
  CONCAT_WS(' ', cust.cstfname, cust.cstlname) AS customer_name,
  color.colorname AS color_name,
  supl.supname AS company_name,
  ship.prc AS price
FROM ship AS ship
JOIN supl AS supl
  ON ship.comid = supl.supid AND supl.supname IN ('Pallette Emporium', 'Rainbow Inc.')
JOIN color AS color
  ON color.b >= 100
  AND color.g >= 100
  AND color.identname = ship.colid
  AND color.r >= 100
JOIN cust AS cust
  ON cust.cstid = ship.cusid
ORDER BY
  5 DESC NULLS LAST
LIMIT 5


In [26]:
print(bc.generate_plan(pydough.to_sql(q)))

CombineStreamsExchange
  BodoPhysicalProject(SHIPMENT_ID=[$0], CUSTOMER_NAME=[CONCAT_WS(' ', $4, $5)], COLOR_NAME=[$3], COMPANY_NAME=[$2], PRICE=[$1])
    BodoPhysicalSort(sort0=[$1], dir0=[DESC-nulls-last], fetch=[5])
      BodoPhysicalProject(SID=[$0], PRC=[$2], SUPNAME=[$3], COLORNAME=[$4], CSTFNAME=[$6], CSTLNAME=[$7])
        BodoPhysicalJoin(condition=[=($5, $1)], joinType=[inner], JoinID=[0])
          BodoPhysicalProject(SID=[$0], CUSID=[$2], PRC=[$3], SUPNAME=[$4], COLORNAME=[$6])
            BodoPhysicalJoin(condition=[=($5, $1)], joinType=[inner], JoinID=[1])
              BodoPhysicalProject(SID=[$0], COLID=[$1], CUSID=[$2], PRC=[$4], SUPNAME=[$6])
                BodoPhysicalJoin(condition=[=($3, $5)], joinType=[inner], JoinID=[2])
                  IcebergToBodoPhysicalConverter
                    IcebergRuntimeJoinFilter(joinIDs=[[0, 1, 2]], equalityColumnsList=[[[2], [1], [3]]], allEqualityKeysReady=[[true, true, true]], nonEqualityColumnsList=[[[], [], []]])
         

The logical plan for this query reveals a few key optimizations BodoSQL is able to perform:

* The BodoSQL planner does filter pushdown. In this example, the planner pushes the filter on color R,G,B values being greater than or equal to 100 into the Iceberg I/O node for the color table. 

* For SQL queries consisting of multiple joins, the BodoSQL planner can reorder joins to minimize the size of intermediate results. In this example, the Shipments table is first joined with smallest table, Suppliers, before joining on the Colors and Customers tables.  

* The BodoSQL planner produces join filters (`IcebergRuntimeJoinFilter`), which are filters applied to the probe side of a join at runtime based on the values seen by the build table at runtime. In this example, a filter for the Shipments table is generated, which selects all shipments where the company key matches the key of either Pallette Emporium or Rainbow Inc. This filter gets pushed into the Iceberg I/O node for the Shipments table. Since Shipments was originally partitioned on company key, BodoSQL only needs to read **40%** of the files in that table for this query. 

In [27]:
pydough.to_df(q)

,SHIPMENT_ID,CUSTOMER_NAME,COLOR_NAME,COMPANY_NAME,PRICE
0,135336,Frank Davis,Rich Brilliant Lavender,Rainbow Inc.,230.9
1,146353,Marcel Jackson,Rich Brilliant Lavender,Rainbow Inc.,230.9
2,16225,Heidi Wilson,Pale Lavender,Rainbow Inc.,230.6
3,89398,Bob Johnson,Pale Lavender,Rainbow Inc.,230.6
4,23408,Oscar Harris,Baby Blue,Rainbow Inc.,230.1


## Cleanup Tables

In [13]:
shutil.rmtree(iceberg_warehouse)